In [1]:
import os
import random
import pygame
import math
from ipapy import UNICODE_TO_IPA

# Setting base directory for .ipynb
BASE_DIR = os.getcwd()

def get_path(relative_path):
    return os.path.join(BASE_DIR, relative_path)

# INITIAL VALUES
invalid_word_message = None
scroll_position = 0
scroll_position_sol = 0
solutions_per_page = 8
solutions_per_page_sol = 13
pygame.double_buffer = True
hexagon_size = 50
earnedpoints = 0
totalpoints = 0
WHITE = (255, 255, 255) # CAN CHANGE THIS FOR DIFF BG COLOR

# EMPTY LISTS
selected_letters = [] # letters user has selected
solutions = [] # solutions the user has entered
hasbeencounted = [] # words that have been added into points
hasbeenconverted = [] # words that have been checked in solutions
solution_pairs = []
spellingbee_pairs = []
spellingbeewords = []
pangram = [] 
user_words = set()
wordstot = []
words = []
ipaletters=[]

# DEFINITIONS OF STATES
STATE_MENU = 0
STATE_GAME = 1
STATE_MENU2 = 2
STATE_SOLUTIONS = 3
current_state = STATE_MENU
was_in_game_state = False

ipaconsonants = ['x', 'ð', 'ʤ', 'd', 'b', 'f', 'g', 'h', 'j', 'k', 'l', 'm', 'n', 'ŋ', 'p', 'r', 's', 'ʃ', 't', 'ʧ', 'θ', 'v', 'w', 'z', 'ʒ']
ipavowels = ['aʊ', 'aɪ', 'ɛ', 'eɪ', 'ɪ', 'ɒ', 'ɑ', 'i', 'oʊ', 'ɔ', 'ɔɪ', 'ʊ', 'u', 'ʌ', 'ə', 'ɪ', 'u','æ']
ipa = ipaconsonants + ipavowels


"""
BEGIN FUNCTIONS
"""
def get_english(value, input_list): # will be used for each value in wordstot to get english transliteration
    if value in input_list:
        index = len(input_list) - input_list[::-1].index(value) - 1
        if index >= 2:
            return input_list[index - 2]
    return None

def get_fullipa(value, input_list): # will be used for each value in wordstot to get full ipa transliteration
    if value in input_list:
        index = len(input_list) - input_list[::-1].index(value) - 1
        if index >= 1:
            return input_list[index - 1]
    return None

def getletters():
    letters = []
    while len(letters) < 10:
        if len(letters) == 1:
            letter = random.choice(ipa)
            letters.append(letter)
        else:
            letter = random.choice(ipa)
            if letter not in letters:
                letters.append(letter)
    global letter1
    letter1 = letters[0]
    return letters

def find_solutions(words, letters, letter1, mode, counter=0):
    from functools import lru_cache

    @lru_cache(None)
    def best_count(position, used, has_required):
        if position == len(word):
            return len(used) if has_required else -1

        best = -1
        for letter in letters:
            if word.startswith(letter, position):
                result = best_count(
                    position + len(letter),
                    frozenset(list(used) + [letter]),
                    has_required or letter == letter1
                )
                best = max(best, result)

        return best

    spellingbeewords = []
    pangram = []

    for word in words:
        if len(word) < 4:
            continue

        best_count.cache_clear()
        if best_count(0, frozenset(), False) >= 0:
            spellingbeewords.append(word) # If all a word's letters are usable it will be added
            if best_count(0, frozenset(), False) >= 7:
                pangram.append(word) # If at least 7 usable letters are in a word it will be added

    if mode == "pangram":
        return pangram
    if mode == "spellingbeewords":
        return spellingbeewords


 
"""
PYGAME BUTTON FUNCTIONS
"""

# draws a hexagonal button
def draw_hexagonal_button(surface, x, y, side_length, text):
    num_sides = 6
    angle = 360 / num_sides
    button_points = []

    for i in range(num_sides):
        x_point = x + side_length * math.cos(math.radians(i * angle))
        y_point = y + side_length * math.sin(math.radians(i * angle))
        button_points.append((x_point, y_point))

    if text == letter1:
        pygame.draw.polygon(surface, (255,217,25), button_points)  # Yellow color
    else:
        pygame.draw.polygon(surface, (220,220,220), button_points)  # Gray color

    font_path = get_path('ipafont.ttf')  # Dynamically generated path
    font = pygame.font.Font(font_path, 36) # font for buttons
    text_surface = font.render(text, True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(x, y))
    surface.blit(text_surface, text_rect)

# checks to see if a point is in a hexagon
def point_in_hexagon(point, x, y, side_length):
    # Calculate the coordinates of the vertices of the hexagon
    angle = 360 / 6  # 60 degrees
    vertices = [(x + side_length * math.cos(math.radians(i * angle)),
                 y + side_length * math.sin(math.radians(i * angle))) for i in range(6)]

    # Check if the point is inside the hexagon
    x, y = point
    odd = False
    i = 0
    j = 5
    for i in range(6):
        xi, yi = vertices[i]
        xj, yj = vertices[j]
        if ((yi < y and yj >= y) or (yj < y and yi >= y)) and (xi + (y - yi) / (yj - yi) * (xj - xi) < x):
            odd = not odd
        j = i
    return odd

"""
PYGAME SCREEN FUNCTIONS
"""

def draw_opening_screen(screen):
    screen.fill((255, 255, 255))  # White background

    # getting parts of fact
    part1 = ' '.join(fact[:6])
    part2 = ' '.join(fact[6:14])
    part3 = ' '.join(fact[14:22])
    part4 = ' '.join(fact[22:30])

    # text to indicate the start of the game
    RalewayPath = get_path('Raleway/static/Raleway-Medium.ttf')
    RalewayMedium = pygame.font.Font(RalewayPath, 24)
    number = str(len(solutions))
    if was_in_game_state:
        text_surface1 = RalewayMedium.render("You've found {} words. Press return to continue.".format(number), True, (0, 0, 0))
        text_rect1 = text_surface1.get_rect(center=(640, 300))
        screen.blit(text_surface1, text_rect1)
    else:
        text_surface1 = RalewayMedium.render("Press return to start.", True, (0, 0, 0))
        text_rect1 = text_surface1.get_rect(center=(640, 300))
        text_surface2 = RalewayMedium.render("Did you know? {}".format(part1), True, (179,149,0))
        text_rect2 = text_surface2.get_rect(center=(640, 340))
        text_surface3 = RalewayMedium.render("{}".format(part2), True, (179,149,0))
        text_rect3 = text_surface3.get_rect(center=(640, 365))
        text_surface4 = RalewayMedium.render("{}".format(part3), True, (179,149,0))
        text_rect4 = text_surface4.get_rect(center=(640, 390))
        text_surface5 = RalewayMedium.render("{}".format(part4), True, (179,149,0))
        text_rect5 = text_surface5.get_rect(center=(640, 415))
        screen.blit(text_surface1, text_rect1)
        screen.blit(text_surface2, text_rect2)
        screen.blit(text_surface3, text_rect3)
        screen.blit(text_surface4, text_rect4)
        screen.blit(text_surface5, text_rect5)

def draw_menu(screen):
    screen.fill(WHITE)

    RalewayPath = get_path('Raleway/static/Raleway-Medium.ttf')  
    RalewayMedium = pygame.font.Font(RalewayPath, 24)
    text_surface1 = RalewayMedium.render("Press [esc] to exit menu.", True, (0, 0, 0))
    text_rect1 = text_surface1.get_rect(center=(640, 130))
    screen.blit(text_surface1, text_rect1)

    # draws "Give Up" and "See Hints" buttons on screen
    pygame.draw.rect(screen, (220,220,220), (570, 175, 140, 50), width=0, border_radius=7)  # Gray color
    text_surface = RalewayMedium.render("Give Up", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(640, 200))
    screen.blit(text_surface, text_rect)

    text_surface = RalewayMedium.render("Key:", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(640, 270))
    screen.blit(text_surface, text_rect)

    # displays key
    font_path = get_path('ipafont.ttf') 
    smallfont = pygame.font.Font(font_path, 20)
    for index, value in enumerate(ipaletters):
        y = 325 + (30 * index)
    pygame.draw.rect(screen, (220,220,220), (375, 305, 530, (y-325)+45), width=0, border_radius=7)  # Gray color
    for index, value in enumerate(ipaletters):
        text_surface = smallfont.render("{}".format(ipaletters[index]), True, (0, 0, 0))
        y = 325 + (30 * index)
        text_rect = text_surface.get_rect(center=(640, y))
        screen.blit(text_surface, text_rect)

def draw_solutions(screen):
    
    screen.fill(WHITE)

    RalewayPath = get_path('Raleway/static/Raleway-Medium.ttf') 
    RalewayMedium = pygame.font.Font(RalewayPath, 24)
    text_surface1 = RalewayMedium.render("Press [esc] to go back to the puzzle.", True, (0, 0, 0))
    text_rect1 = text_surface1.get_rect(center=(640, 50))
    screen.blit(text_surface1, text_rect1)

    text_surface2 = RalewayMedium.render("Today's solutions are:", True, (0, 0, 0))
    text_rect2 = text_surface2.get_rect(center=(640, 95))
    screen.blit(text_surface2, text_rect2)
    
    y2 = 135

    font_path = get_path('ipafont.ttf') 
    smallfont = pygame.font.Font(font_path, 20)
    for i in range(start_index_sol, end_index_sol):
        if i < len(spellingbee_pairs):
            ipasolution, englishsolution = spellingbee_pairs[i]

            if i % 3 == 1:
                x = 640
            elif i % 3 == 2:
                x = 880
            else:
                x = 400

            if ipasolution.replace("ˈ", "").replace("ˌ", "") not in pangram:
                text_surface = smallfont.render('{} ({})'.format(ipasolution, englishsolution), True, (0, 0, 0))
            else:
                text_surface = smallfont.render('{} ({})'.format(ipasolution, englishsolution), True, (179, 149, 0))

            text_rect = text_surface.get_rect(center= (x, y2))
            screen.blit(text_surface, text_rect)

            if i % 3 == 2:
                y2 += 40

def draw_game_screen(screen):
    
    # creates rectangle for displayed list of solutions in right corner
    font_path = get_path('ipafont.ttf')  
    font = pygame.font.Font(font_path, 24)  # font
    solutions_rect = pygame.Rect(900, 50, 300, 600) 
    x = solutions_rect.left  # starting position for rendering the text
    y = solutions_rect.top

    # getting solution_pairs from solutions
    for solution in solutions:
        if solution != get_fullipa(solution, wordstot):
            if (get_fullipa(solution, wordstot), get_english(solution, wordstot)) not in solution_pairs:
                solution_pairs.append((get_fullipa(solution, wordstot), get_english(solution, wordstot)))
        else:
            if (solution, get_english(solution, wordstot)) not in solution_pairs:
                solution_pairs.append((solution, get_english(solution, wordstot)))

    # Render the solutions within the visible portion
    for i in range(start_index, end_index):
        if i < len(solution_pairs):
            solution, english_value = solution_pairs[i]
            solutions_surface = font.render(f'{solution} ({english_value})', True, (0, 0, 0))
            solutions_rect = solutions_surface.get_rect(topright=(900, y))  # Adjust the x coordinate for rendering text
            screen.blit(solutions_surface, solutions_rect.topright)
            y += solutions_rect.height  # Adjust the vertical offset for the next line

    # draws hexagons on screen
    draw_hexagonal_button(screen, 500, 300, hexagon_size, letters[1])
    draw_hexagonal_button(screen, 500, 400, hexagon_size, letters[0]) #required letter
    draw_hexagonal_button(screen, 500, 500, hexagon_size, letters[3])
    draw_hexagonal_button(screen, 500, 600, hexagon_size, letters[4])
    draw_hexagonal_button(screen, 413, 350, hexagon_size, letters[5])
    draw_hexagonal_button(screen, 413, 450, hexagon_size, letters[6])
    draw_hexagonal_button(screen, 413, 550, hexagon_size, letters[7])
    draw_hexagonal_button(screen, 587, 350, hexagon_size, letters[8])
    draw_hexagonal_button(screen, 587, 450, hexagon_size, letters[9])
    draw_hexagonal_button(screen, 587, 550, hexagon_size, letters[2])

    # draws "Clear" and "Enter" buttons on screen
    pygame.draw.rect(screen, (220,220,220), (25, 580, 100, 50), width=0, border_radius=7)  # Gray color
    RalewayPath = get_path('Raleway/static/Raleway-Medium.ttf')  
    RalewayMedium = pygame.font.Font(RalewayPath, 24)
    text_surface = RalewayMedium.render("Clear", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(75, 605))
    screen.blit(text_surface, text_rect)

    pygame.draw.rect(screen, (220,220,220), (25, 645, 100, 50), width=0, border_radius=7)  # Gray color
    text_surface = RalewayMedium.render("Enter", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(75, 670))
    screen.blit(text_surface, text_rect)

    # draws a menu button
    pygame.draw.rect(screen, (220,220,220), (25, 25, 100, 50), width=0, border_radius=7)  # Gray color
    text_surface = RalewayMedium.render("Menu", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(75, 50))
    screen.blit(text_surface, text_rect)

    # draws a text box with additional info
    RalewayPath = get_path('Raleway/static/Raleway-Medium.ttf')
    Ralewaysmall = pygame.font.Font(RalewayPath, 18)
    
    pygame.draw.rect(screen, (220,220,220), (900, 425, 355, 275), width=0, border_radius=7)  # Gray color
    text_surface = RalewayMedium.render("Welcome to the IPA Bee!", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 460))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render("The International Phonetic Alphabet is", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 510))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render("a standardized representation of", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 530))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render("speech in written form. It's used in a", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 550))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render(" variety of professions, from linguistics", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 570))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render(" to opera, to transcribe lexical and", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 590))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render("prosodic sounds in oral language.", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 610))
    screen.blit(text_surface, text_rect)
    text_surface = Ralewaysmall.render("Broad transcriptions are used for clarity.", True, (0, 0, 0))
    text_rect = text_surface.get_rect(center=(1077, 670))
    screen.blit(text_surface, text_rect)

    # displays selected text
    font = pygame.font.Font(font_path, 36) 
    selected_text = "".join(selected_letters)
    selected_text_surface = font.render(selected_text, True, (0, 0, 0))
    selected_text_rect = selected_text_surface.get_rect(center=(500, 200))  # can adjust position as needed
    screen.blit(selected_text_surface, selected_text_rect)

    #invalid message display
    # change later to be in Raleway except for the variable
    global invalid_word_message
    if invalid_word_message:
        # RalewayPath = get_path('Raleway/static/Raleway-Medium.ttf')
        # RalewayMedium = pygame.font.Font(RalewayPath, 24)
        invalid_text_surface = font.render(invalid_word_message, True, (0, 0, 0))
        invalid_text_rect = invalid_text_surface.get_rect(center=(500, 80))
        screen.blit(invalid_text_surface, invalid_text_rect)
        if current_time - invalid_word_message_start_time >= 4000:  # 4 seconds
            invalid_word_message = None
    
    global earnedpoints
    for i in solutions:
        if i not in hasbeencounted:
            hasbeencounted.append(i)
            if len(i) == 4:
                earnedpoints += 1
            elif i in pangram:
                earnedpoints += (2 * len(i))
            else:
                earnedpoints += (len(i))

    # displays progress
    progress = (earnedpoints / totalpoints)
    progresslength = progress * 200
    pygame.draw.line(screen, (0, 0, 0), (400,30), (600,30), width=3)
    pygame.draw.line(screen, (0, 180, 0), (400,30), (400 + progresslength,30), width=3)

    # ends the game if all the words have been found
    if len(solutions) == len(spellingbeewords):
        screen.fill(WHITE)
        endgame_surface = RalewayMedium.render("You've reached Queen Bee!", True, (0, 0, 0))
        endgame_rect = endgame_surface.get_rect(center=(640, 340))
        screen.blit(endgame_surface, endgame_rect)
        endgame_surface2 = RalewayMedium.render("Congratulations on completing the game.", True, (0, 0, 0))
        endgame_rect2 = endgame_surface2.get_rect(center=(640, 380))
        screen.blit(endgame_surface2, endgame_rect2)

"""
*************
END FUNCTIONS
*************
"""

# READING FILE CONTENT - IPA WORD LIST WITH ENGLISH EQUIVALENTS AND IPA SIMPLIFIED VERSIONS
file_path = get_path("ipaengwords.txt")  # Dynamically generated path
with open(file_path, "r") as txt_file:
    file_content = txt_file.readlines()

# READING FILE CONTENT - LINGUISTICS FACTS
file_path2 = get_path("linguisticsfacts.txt")  # Dynamically generated path
with open(file_path2, "r") as txt_file:
    file_content2 = txt_file.readlines()
linguisticsfacts =[]
for i in file_content2:
    i.split("\n",1)[0]
    linguisticsfacts.append(i)
factword = random.choice(linguisticsfacts)
fact = factword.split()

# DATA CLEANING OF WORDLIST
for i, value in enumerate(file_content):
    sep = '\n'
    m = value.split(sep, 1)[0]
    wordstot.append(m)
    if (i + 1) % 2 == 0:
        j = m.replace("ˈ", "")
        z = j.replace("ˌ", "")
        wordstot.append(z)

# ADDING ONLY SIMPLIFIED IPA WORDS TO A LIST
for index, value in enumerate(wordstot):
    if (index + 1) % 3 == 0 and value not in words:
        words.append(value)

# getting letters that have the right number of solutions
attempts = 0
max_attempts = 10000

while len(pangram) == 0 or len(spellingbeewords) < 10:
    letters = getletters()
    spellingbeewords = find_solutions(words, letters, letter1, "spellingbeewords")
    pangram = find_solutions(words, letters, letter1, "pangram")
    attempts += 1
    if attempts >= max_attempts:
        raise RuntimeError("Could not generate a valid puzzle. Check that the IPA word list file exists.")


total_words = len(spellingbeewords)
found_words = 0

# PRINTS OUTPUTS TO TERMINAL
print("Here are today's usable sounds:")
print(letters)
print("This is the required sound:")
print(letter1)
print("Here is the full list of today's answers:")
print(spellingbeewords)
print("Here are today's pangrams:")
print(pangram)

# CONVERTS EACH UNICODE LETTER TO IPA DESCRIPTION - USED IN THE MENU SECTION TO GIVE MORE INFO ABOUT EACH LETTER
for c in letters:
    for letter in c:
        i = UNICODE_TO_IPA[letter]
        ipaletters.append(letter + " - " + i.name)

# CREATING PAIRS OF IPA SOLUTIONS AND THEIR ENGLISH EQUIVALENTS
for solution in spellingbeewords:
    if solution != get_fullipa(solution, wordstot):
        if (get_fullipa(solution, wordstot), get_english(solution, wordstot)) not in spellingbee_pairs:
            spellingbee_pairs.append((get_fullipa(solution, wordstot), get_english(solution, wordstot)))
    else:
        if (solution, get_english(solution, wordstot)) not in solution_pairs:
            spellingbee_pairs.append((solution, get_english(solution, wordstot)))

# INITIAL PYGAME LAUNCH
pygame.init()
screen = pygame.display.set_mode((1280, 720))
pygame.display.set_caption("IPA Spelling Bee") # name of the game
clock = pygame.time.Clock()
running = True
in_game = True

# COMPUTES TOTAL AMOUNT OF POSSIBLE POINTS
for i in spellingbeewords:
    if len(i) <= 4:
        totalpoints += 1
    elif i in pangram:
        totalpoints += (len(i) * 2)
    else:
        totalpoints += len(i)

quotient = len(spellingbee_pairs) // 3  # integer division to get the greatest number of times 3 goes into
remainder = len(spellingbee_pairs) % 3  # calculate the remainder
if remainder > 0:
    quotient += 1

max_scroll_position_sol = max(0, (quotient - solutions_per_page_sol))

"""
START OF WHILE LOOP IN PYGAME
*****************************
"""
while running:
    current_time = pygame.time.get_ticks() # tracks the time for all of the messages
    is_active = pygame.display.get_active() # tells whether the display is active for the away message

    screen.fill(WHITE)

    start_index = scroll_position
    start_index_sol = scroll_position_sol
    end_index = scroll_position + solutions_per_page # indices for scrolling 
    end_index_sol = min(len(spellingbee_pairs), start_index_sol + (3 * solutions_per_page_sol)) #end_index_sol = scroll_position_sol + (3 * solutions_per_page_sol) # indices for scrolling

    max_scroll_position = max(0, len(solution_pairs) - solutions_per_page)

    if not is_active: # checks to see if it is active
        current_state = STATE_MENU
        was_in_game_state = True

    # pygame.QUIT event means the user clicked X to close their window
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.MOUSEBUTTONDOWN:
            if event.button == 4:  # scroll up (mouse wheel up)
                if current_state == STATE_GAME:
                    scroll_position = max(0, scroll_position - 1)
                if current_state == STATE_SOLUTIONS:
                    scroll_position_sol = max(0, scroll_position_sol - 3)
            elif event.button == 5:  # scroll down (mouse wheel down)
                if current_state == STATE_GAME:
                    scroll_position = min(max_scroll_position, scroll_position + 1)
                if current_state == STATE_SOLUTIONS:
                    scroll_position_sol = min(max_scroll_position_sol, scroll_position_sol + 3)
            if point_in_hexagon(event.pos, 500, 400, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letter1)
            if point_in_hexagon(event.pos, 500, 300, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[1])
            if point_in_hexagon(event.pos, 500, 500, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[3])
            if point_in_hexagon(event.pos, 500, 600, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[4])
            if point_in_hexagon(event.pos, 413, 350, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[5])
            if point_in_hexagon(event.pos, 413, 450, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[6])
            if point_in_hexagon(event.pos, 413, 550, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[7])
            if point_in_hexagon(event.pos, 587, 350, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[8])
            if point_in_hexagon(event.pos, 587, 450, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[9])
            if point_in_hexagon(event.pos, 587, 550, hexagon_size): # check if a hexagonal button was clicked
                selected_letters.append(letters[2])
            if 25 <= event.pos[0] <= 125 and 580 <= event.pos[1] <= 630: # if we hit the clear button
                selected_letters = []
            if 25 <= event.pos[0] <= 125 and 645 <= event.pos[1] <= 695: # if we hit the enter button
                attempt = "".join(selected_letters)
                selected_letters = []
                if attempt in spellingbeewords and letter1 in attempt:
                    if attempt in solutions:
                        invalid_word_message = f'"{attempt}" has already been found.'
                        invalid_word_message_start_time = current_time    
                    elif attempt in pangram:
                        invalid_word_message = f"You've found a pangram!"
                        invalid_word_message_start_time = current_time  
                        solutions.append(attempt)
                    else:
                        solutions.append(attempt)
                elif letter1 not in attempt:
                    invalid_word_message = f'"{attempt}" does not contain "{letter1}"—try again!'
                    invalid_word_message_start_time = current_time                                         
                else:
                    invalid_word_message = f'"{attempt}" is not a valid word—try again!'
                    invalid_word_message_start_time = current_time
            if 25 <= event.pos[0] <= 125 and 25 <= event.pos[1] <= 75:
                current_state = STATE_MENU2
            if 570 <= event.pos[0] <= 710 and 175 <= event.pos[1] <= 225:
                if current_state == STATE_MENU2:
                    current_state = STATE_SOLUTIONS
        elif event.type == pygame.KEYDOWN:
            if current_state == STATE_MENU:
                if event.key == pygame.K_RETURN:
                    # transition to the game state when return is pressed
                    current_state = STATE_GAME
                    is_active = True
            else:
                if event.key == pygame.K_1:
                    print("{} max_scroll_position_sol".format(max_scroll_position_sol))
                    print("{} scroll_position_sol".format(scroll_position_sol))
                    print(start_index_sol)
                    print(end_index_sol)
                    print("{} quotient".format(quotient))
                if event.key == pygame.K_ESCAPE:
                    if current_state == STATE_MENU2:
                        current_state = STATE_GAME
                    if current_state == STATE_SOLUTIONS:
                        current_state = STATE_GAME
                    elif current_state == STATE_GAME:
                        selected_letters = []
                if event.key == pygame.K_RETURN:
                    attempt = "".join(selected_letters)
                    selected_letters = []
                    if attempt in spellingbeewords and letter1 in attempt:
                        if attempt in solutions:
                            invalid_word_message = f'"{attempt}" has already been found.'
                            invalid_word_message_start_time = current_time    
                        elif attempt in pangram:
                            invalid_word_message = f"You found a pangram!"
                            invalid_word_message_start_time = current_time  
                            solutions.append(attempt)
                        else:
                            solutions.append(attempt)
                    elif letter1 not in attempt:
                        invalid_word_message = f'"{attempt}" does not contain "{letter1}"—try again!'
                        invalid_word_message_start_time = current_time                                         
                    else:
                        invalid_word_message = f'"{attempt}" is not a valid word—try again!'
                        invalid_word_message_start_time = current_time

    if current_state == STATE_MENU:
        draw_opening_screen(screen)
    elif current_state == STATE_GAME:
        draw_game_screen(screen)
    elif current_state == STATE_MENU2:
        draw_menu(screen)
    elif current_state == STATE_SOLUTIONS:
        draw_solutions(screen)

    pygame.display.flip()  # updates display

    clock.tick(60)  # frame rate @ 60 FPS

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
Here are today's usable sounds:
['æ', 'k', 'l', 'ə', 'p', 's', 'g', 'ʧ', 'd', 'n']
This is the required sound:
æ
Here is the full list of today's answers:
['æsəd', 'ædəl', 'əlæs', 'ælgəl', 'ælpækə', 'ənæləgəs', 'ænəs', 'ænəl', 'ænəld', 'æpəl', 'əskæns', 'æspən', 'əsæsən', 'æksəs', 'æksəl', 'kækəl', 'kælə', 'kæləs', 'kæləsnəs', 'kənæl', 'kænkæn', 'kænsəl', 'kændəl', 'kænd', 'kænən', 'kæpsəl', 'kæsk', 'kæsək', 'kæsəl', 'ʧæns', 'ʧænəl', 'ʧæpəl', 'ʧæplən', 'klæk', 'klæd', 'klæn', 'klæp', 'klæsp', 'klæs', 'klæsləs', 'kəlæps', 'dæns', 'kænə', 'kæpə', 'næpsæk', 'lægd', 'læns', 'lænd', 'læps', 'læks', 'nægd', 'pædəl', 'pædək', 'pæləs', 'pæləd', 'pændə', 'pænəl', 'pænd', 'pæskæl', 'plæd', 'plæn', 'plænd', 'plæk', 'sædən', 'sædəl', 'sædnəs', 'sægd', 'sæləd', 'sænd', 'sændəl', 'sæks', 'skæd', 'skæləp', 'skælp', 'skælpəl', 'skæn', 'skændəl', 'skændələs', 'skænd', 'sədæn', 'skəlkæp', 'sl